In [8]:
import json
from gliotrace.gliotrace_class import GlioTrace

with open("stackable.json", "r") as f:
    stackfile = json.load(f)

metadata = "metadata.csv"
detection_sensitivity = 0
channel_roles = {"blue": 0, "green": 1, "red": 2}

fcols = ["delta_t", "tme_label", "state_label", "sum_green", "growth_rate",
    "speed", "direction", "roi_speed", "sin", "cos", "roi_sin", "roi_cos",
    "roi_direction", "weighted_sin", "weighted_cos", "polarization",
    "cos_sim", "faster", "directional_change", "speed_change",
    "roi_directional_change", "roi_speed_change", "vascular_distance",
    "adMAD", "speed_treat", "direction_treat", "roi_speed_treat",
    "roi_direction_treat", "polarization_treat", "cos_sim_treat", "faster_treat",
    "directional_change_treat", "speed_change_treat", "roi_directional_change_treat",
    "roi_speed_change_treat", "growth_rate_treat", "sum_green_treat"
]



In [ ]:
# The different modes that gliotrace can now run with:

# ------------------------------------------
# To run all patients all sets, no treatment
# ------------------------------------------

gliobj_3013_ctrl = Gliotrace(stackfile=stackfile, 
                             metadata=metadata, 
                             detection_sensitivity=detection_sensitivity,
                             fcols=fcols,
                             verbose=True)

# ----------------------------------------------------------------------
# To run all patients, but specify subset for some pations, no treatment
# ----------------------------------------------------------------------
sets_by_patient = {
    "andre": [1,2,3] # <-- This says that for patient andre the sets 1,2,3 should be used, for everyone else every set
}

gliobj_3013_ctrl = Gliotrace(stackfile=stackfile, 
                             metadata=metadata, 
                             detection_sensitivity=detection_sensitivity,
                             fcols=fcols,
                             sets_by_patient=sets_by_patient, # <--- added here
                             verbose=True)

# ----------------------------------------------------------------------
# To run some patients, all sets, no treatment
# ----------------------------------------------------------------------
patient_id = ["andre", "linnea", "madde"]

gliobj_3013_ctrl = Gliotrace(stackfile=stackfile, 
                             metadata=metadata, 
                             detection_sensitivity=detection_sensitivity,
                             fcols=fcols,
                             patient_id=patient_id, # <---- Added here
                             verbose=True)


# ----------------------------------------------------------------------
# To run some patients, some sets, no treatment
# ----------------------------------------------------------------------
patient_id = ["andre", "linnea"]
sets_by_patient = {
    "andre": [1,2,3] # <-- This says that for patient andre the sets 1,2,3 should be used, for linnea all sets
}

# NOTE: If you want to specify patient_id and sets, then the sets patients (here 'andre') must also be in patient_id

gliobj_3013_ctrl = Gliotrace(stackfile=stackfile, 
                             metadata=metadata, 
                             detection_sensitivity=detection_sensitivity,
                             fcols=fcols,
                             patient_id=patient_id,          # <--- Here
                             sets_by_patient=sets_by_patient # <--- Here
                             verbose=True)


# ----------------------------------------------------------------------
# To run some patients, some sets, and treatment
# ----------------------------------------------------------------------
patient_id = ["andre", "linnea"]
sets_by_patient = {
    "andre": [1,2,3]
}

# To further specify treatment there are two possibilites:
treatment = ["drug", None]   # <--- This says "drug" is the treatment and no filtering on dosage
treatment = ["drug", "half"] # <--- This says "drug" is the treatment and dose = "half"

# Whenever treatment is specified, Perturbation has to be specified as well
# Perturbation acts as the control and is a string
perturbation = "control"

gliobj_3013_ctrl = Gliotrace(stackfile=stackfile, 
                             metadata=metadata, 
                             detection_sensitivity=detection_sensitivity,
                             fcols=fcols,
                             patient_id=patient_id,          
                             sets_by_patient=sets_by_patient,
                             treatment=treatment,        # <--- Here
                             perturbation=perturbation,  # <--- Here
                             verbose=True)

# ---------------------------------------
# Question that may arise (not exhaustive):
# ----------------------------------------

# Patient ids are all assumed to be strings
# Sets are ints and require a patient
# Treatment and dosage are strings
# Treatment and perturbation are both based on the "perturbation" column in metadata, dose on 'dose' column
# Not specifying pertubation means no filtering there
# Filtering can be done by pertubation without treatment
# If one wants to do multi filtering i.e patients, sets and treatments, warnings are given if a combination does not exist


In [14]:
patient_id = ["patientx", "patienty"]
treatment = ["grain", "0.5"]  
perturbation = "placebo" 
hmm_params={"glm_iter":1000}

gliobj_3013_ctrl = GlioTrace(stackfile=stackfile, 
                             metadata=metadata, 
                             detection_sensitivity=detection_sensitivity,
                             fcols=fcols,
                             control="brain", 
                             treatment=treatment,        
                             hmm_param=hmm_params,
                             verbose=True)


=== Gliotrace configuration ===
Number of stacks        : 4
Scope                   :
  Exps                  : [64, 148]
Detection sensitivity   : 0.0
Channel roles           : {'blue': 'none', 'green': 'gbm', 'red': 'vasc'}
Feature columns         : ['speed']
HMM parameters          : {'em_iter': 1000, 'penalty': 0.01, 'glm_iter': 1000, 'eps': 1e-06}
Control                 : brain
Treatment               : grain (dose=0.5)
Stacks (control/treat)  : 3 / 1



In [3]:
# Run tracking on the stacks as specified in stackfile
gliobj_3013_ctrl.run_tracking()
gliobj_3013_ctrl.glm_hmm()

Building slice statistics table...
Number of stacks: 4
Processing stacks. Format: completed / total [elapsed time < time remaining, seconds per stack].


Tracking and classifying cells: 100%|██████████| 4/4 [00:06<00:00,  1.61s/stack]


--- Creating features ---
--- Intialize transitions ---
--- Final Preperations for GLM-HMM ---
--- Running GLM-HMM ---


Performing CNN-HMM - GLM algorithm:   1%|          | 12/1000 [00:00<00:46, 21.40EM-step/s]

Early stop: no improvement for 10 steps. Returning best model.
--- Computing Viterbi Paths ---


In [ ]:
# So now I want to look at summaries, what can i do:

# Case 1: Metadata does not have patient id, both treatment and control:
# output format dictionary: dict[experiment] = dataframe time x roi
ex1 = gliobj_3013_ctrl.generate_summary_statistics(fcol="speed", mode="mean")

# Case 2: Metadata does not have patient id, treatment:
# output format dictionary: dict[experiment] = dataframe time x roi
ex2 = gliobj_3013_ctrl.generate_summary_statistics(fcol="speed", treatment=False, mode="mean")
ex3 = gliobj_3013_ctrl.generate_summary_statistics(fcol="speed", treatment=True, mode="mean")
ex4 = gliobj_3013_ctrl.generate_summary_statistics(fcol="speed", mode="mean") # <--- both

# Case 3: Metadata has patient id not sets, no treat
# output format dictionary: dict[experiment] = dataframe time x roi
ex5 = gliobj_3013_ctrl.generate_summary_statistics(fcol="speed", patient_id="patientx", treatment=False, mode="mean")

# Case 4: Metadata has patient id and sets, no treat
# output format dictionary: dict[experiment] = dataframe time x roi
ex5 = gliobj_3013_ctrl.generate_summary_statistics(fcol="speed", patient_id="patientx", set_id=1, treatment=False, mode="mean")


In [11]:
ex5 = gliobj_3013_ctrl.generate_summary_statistics(fcol="speed", patient_id="patientx", set_id=1, treatment=False, mode="mean")

experiments = list(ex5.keys())

ex5[experiments[0]]

roi,1,2,3
time,,,
0,NaN,NaN,NaN
1,0.948393,1.661935,2.264217
2,2.777778,1.202341,2.572554
3,1.291745,1.449709,3.849572
4,0.523783,2.092697,3.195490


In [14]:
# To iterate over patients
print(f"All patients: {gliobj_3013_ctrl.patients()}")
print(f"Patients with some treated experiments: {gliobj_3013_ctrl.patients(treated=True)}")
print(f"Patients with some control experiments: {gliobj_3013_ctrl.patients(treated=False)}")

# To iterate over sets
patient = "patientx"
print(f"All sets for {patient}: {gliobj_3013_ctrl.sets(patient=patient)}")
print(f"{patient}'s sets with treated: {gliobj_3013_ctrl.sets(patient=patient, treated=True)}")
print(f"{patient}'s sets with control: {gliobj_3013_ctrl.sets(patient=patient, treated=False)}")

# Example for iterating over all control and treatment (simply add the treated to sets and patients for that version)
for patient in gliobj_3013_ctrl.patients():
    for set in gliobj_3013_ctrl.sets(patient=patient):
        exp = gliobj_3013_ctrl.generate_summary_statistics(fcol="adMAD", patient_id=patient, set_id=set)
        print(f"Checking patient {patient}, set {set}, experiments {exp.keys()}")

All patients: ['patientx', 'patienty']
Patients with some treated experiments: ['patienty']
Patients with some control experiments: ['patientx']
All sets for patientx: [1]
patientx's sets with treated: []
patientx's sets with control: [1]
Checking patient patientx, set 1, experiments dict_keys([64])
Checking patient patienty, set 2, experiments dict_keys([65])


In [15]:
# Everything else is as usual
gliobj_3013_ctrl.A_global
gliobj_3013_ctrl.gammas
# etc...

,exp,roi,cellID,t,Branching,Diffuse translocation,Junk,Locomotion,Perivascular translocation,Round
0,64,1,0,0,1.431539e-09,0.051591,3.136453e-06,2.318220e-07,1.174329e-11,0.948406
1,64,1,0,1,1.045738e-08,0.023847,1.010957e-08,3.033194e-08,1.102511e-08,0.976153
2,64,1,0,2,6.625970e-09,0.014995,1.102466e-08,2.709631e-08,1.244659e-08,0.985005
3,64,1,0,3,1.495004e-08,0.009208,3.630395e-08,7.656839e-07,1.020240e-08,0.990791
4,64,1,1,0,1.937448e-10,0.000796,1.215172e-06,1.890525e-07,4.740732e-12,0.999203
...,...,...,...,...,...,...,...,...,...,...
99,65,1,10,2,1.893525e-07,0.994102,2.589045e-07,1.005439e-06,2.357560e-08,0.005896
100,65,1,13,0,6.838075e-09,0.802128,6.456715e-05,1.064500e-07,2.444394e-10,0.197807
101,65,1,13,1,2.355427e-07,0.798668,1.914416e-05,4.017139e-07,5.241639e-08,0.201312
102,65,1,14,0,9.559325e-10,0.997170,5.109406e-06,6.006938e-09,4.838753e-11,0.002825


In [ ]:
# Output path for video is optional (a directory is created if non provided)
gliobj.generate_video(exp=1, roi=3, output="/Users/madsk418/Desktop/label_test2/videos_small/")
#gliobj.generate_video_compare(exp=148, roi=1)

In [ ]:
# ------ Check permutation ---------
gliobj_3013_ctrl.compare_hmm_and_cnn_class(threshold=0.9)

In [ ]:
# ---- Compare results ------
print(f"First run columns {cols1}")
print(f"Second run columns {cols2}")

print(f"First pi (state distribution) \n {pi_1}")
print(f"Second pi (State distribution) \n {pi_2}")

print(f"A global 1 \n {A_global_1}")
print(f"A global 2 \n {A_global_2}")
